# 01 Build Local Database

This notebook builds the local DuckDB warehouse used in this analysis.

It uses the same logic as `scripts/build_local_duckdb.py`, with the steps shown in notebook form.

## Configuration

- Set `DOWNLOAD_FROM_KAGGLE = True` if the CSV files are not already present in `data/`.
- Leave `FORCE_REBUILD = True` if you want to overwrite the existing DuckDB file.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import duckdb
import pandas as pd

from scripts.build_local_duckdb import (
    REQUIRED_FILES,
    ensure_data_files,
    read_csv,
    clean_players,
    clean_team_stats,
    clean_player_stats,
    create_raw_tables,
    build_star_schema,
)


In [ ]:
DATA_DIR = ROOT / "data"
DB_PATH = DATA_DIR / "nba_analytics.duckdb"
DOWNLOAD_FROM_KAGGLE = False
FORCE_REBUILD = True

DATA_DIR, DB_PATH

In [ ]:
files = ensure_data_files(DATA_DIR, download=DOWNLOAD_FROM_KAGGLE)
pd.DataFrame({"required_file": list(REQUIRED_FILES), "path": [str(files[name]) for name in REQUIRED_FILES]})

In [ ]:
if DB_PATH.exists() and FORCE_REBUILD:
    DB_PATH.unlink()

team_df = clean_team_stats(read_csv(files["TeamStatistics.csv"], parse_dates=["gameDateTimeEst"]))
player_stats_df = clean_player_stats(read_csv(files["PlayerStatistics.csv"], parse_dates=["gameDateTimeEst"]))
players_df = clean_players(read_csv(files["Players.csv"], parse_dates=["birthdate", "birthDate"]))

print(team_df.shape, player_stats_df.shape, players_df.shape)

In [ ]:
con = duckdb.connect(str(DB_PATH))
create_raw_tables(con, team_df, player_stats_df, players_df)
build_star_schema(con)
print(f"Built local DuckDB database at {DB_PATH}")

In [ ]:
summary = con.execute(
    """
    SELECT * FROM (
        SELECT 'team_stats_raw' AS table_name, COUNT(*) AS row_count FROM team_stats_raw
        UNION ALL
        SELECT 'player_stats_raw', COUNT(*) FROM player_stats_raw
        UNION ALL
        SELECT 'players_raw', COUNT(*) FROM players_raw
        UNION ALL
        SELECT 'players', COUNT(*) FROM players
        UNION ALL
        SELECT 'teams', COUNT(*) FROM teams
        UNION ALL
        SELECT 'games', COUNT(*) FROM games
        UNION ALL
        SELECT 'dates', COUNT(*) FROM dates
        UNION ALL
        SELECT 'player_stats_fact', COUNT(*) FROM player_stats_fact
        UNION ALL
        SELECT 'team_stats_fact', COUNT(*) FROM team_stats_fact
    )
    ORDER BY table_name
    """
).fetchdf()
summary

In [ ]:
con.execute("SHOW TABLES").fetchdf()

## Next Step

Open `02_validation.ipynb` to run read-only checks against the built DuckDB warehouse.